# LangChain 실험 플레이그라운드 — 사용법

밈/신조어 키워드에 대해 질문 -> 벡터 검색(dense/sparse/융합) -> 프롬프트 조립 -> LLM 답변까지,
각 셀을 독립적으로 재실행하며 실험하기 위한 노트북입니다.

**공유 로직**: facet 검색·병합·프롬프트 조립의 핵심 로직은 `analysis/rag_pipeline.py`의 공유 함수(`facet_search`/`build_facet_prompt` 등)로 추출돼 있습니다 — 프로덕션(`rag_main.py`의 facet 모드)과 이 노트북(4번 셀, 8번 셀)이 같은 코드를 씁니다. 이 노트북은 그 함수들을 파라미터를 바꿔가며 호출하는 실험 표면입니다.

## 실행 전 준비

이 노트북은 기본적으로 **EC2 서버(진짜 운영 데이터)** 를 봅니다. 그러려면 별도 터미널에서 SSH 터널이 먼저 떠 있어야 합니다:
```
ssh -i ~/.ssh/id_ed25519 -L 27017:localhost:27017 -L 36333:localhost:16333 ec2-user@100.29.36.216
```
(mongo는 27017 그대로, qdrant는 EC2용으로 로컬 `36333`을 씀 — `6333`은 Windows 예약 포트라 로컬에서 못 열고, `16333`은 로컬 docker qdrant가 이미 쓰고 있어서 겹치지 않게 분리했습니다. 원격(EC2) 쪽 타겟도 `6333`이 아니라 `16333`인데, EC2의 `mimori-qdrant` 컨테이너도 로컬과 동일하게 호스트 16333 → 컨테이너 6333으로 포트가 매핑돼 있기 때문입니다 — EC2 자신도 6333을 직접 안 듣습니다. 타겟을 `localhost:6333`으로 잘못 잡으면 SSH 세션은 붙지만 `channel N: open failed: connect failed: Connection refused`가 뜹니다.)

## 실행 순서

1번(환경설정) → 2번(키워드 선택) → 2.5번(유행 상태, 선택) → 3번(facet 설정+임베딩) → 4번(검색) → 5번(검색결과 확인) → 6번(프롬프트) → 7번(LLM 호출) 순서로 **한 번 끝까지** 실행하세요.

그다음부터는 **3번 ~ 7번만** 값을 바꿔가며 반복 재실행하면 됩니다 (1, 2번은 한 번만 실행하면 됨).

**8번(여러 키워드 일괄 검증)**과 **9번(저장된 기록 비교)**은 2~7번의 단일 키워드 실험과 독립적입니다. 1번 셀만 실행된 상태면 바로 실행 가능하고, 지금 설정이 "야르" 하나가 아니라 여러 키워드에서도 노이즈를 잘 줄이는지 표로 확인할 수 있습니다. 8번을 실행할 때마다 결과가 `eval_runs.jsonl`에 누적되고, 9번에서 그 기록을 시간순으로 비교합니다.

## 셀별로 바꿀 수 있는 값

| 셀 | 변수 | 용도 |
|---|---|---|
| 1. 환경설정 | `USE_EC2` | `True`=EC2 진짜 데이터(터널 필요), `False`=로컬 docker 테스트 데이터 |
| 2. 키워드 선택 | `KEYWORD` | 분석할 밈/신조어. 셀 실행하면 임베딩 완료된 키워드 목록이 출력됨 |
| 2.5. 유행 상태 | `INCLUDE_TREND` | 네이버 데이터랩 유행 상태를 프롬프트에 넣을지 여부 |
| 3. facet 설정+임베딩 | `FACET_CONFIG` | 밈을 의미/유행_이유/사용법/사용자층 네 각도로 나눠 묻는 설정. facet마다 질문 문구/`top_k`/`min_length`/`over_fetch_factor`/`min_dense_score`/`max_per_source` 조절 가능 |
| 4. 검색 파라미터 | `SOURCES` | 검색 대상 소스 제한(모든 facet 공통). 예: `["dcinside","natepann","namuwiki"]`로 tavily/youtube 제외. `None`=전체 |
| 6. 프롬프트 작성 | `analysis/rag_facet_prompt_template.md` | LLM에게 보낼 지시문 파일 — 프로덕션과 공유하는 정식 템플릿. 파일을 수정하고 셀만 재실행하면 반영됨 (`{keyword}`/`{trend_info}`/`{context}` 자리표시자는 유지) |
| 7. LLM 호출 | `MODEL`/`TEMPERATURE`/`TOP_P` | 같은 프롬프트로 답변이 어떻게 달라지는지 비교 |
| 8. 일괄 검증 | `TEST_KEYWORDS`/`EVAL_SOURCES` | 여러 키워드에 대해 노이즈 비율(⚠️%)을 한 번에 비교하고 `eval_runs.jsonl`에 기록 저장 |
| 9. 기록 비교 | `print_eval_history(last_n=N)` | 저장된 실행 기록을 키워드×시각 표로 비교 |

## 5번 셀(검색결과 확인) 읽는 법

- `[facet1,facet2]`: 그 청크가 어느 facet(들)에서 뽑혔는지
- `⚠️키워드 없음(...)`: 길이/키워드 검증을 통과한 청크가 부족해 검증 실패작으로 백필된 경우 — 자주 뜨면 해당 facet의 `over_fetch_factor`를 높이거나 `min_length`를 낮춰볼 것

## 알려진 이슈

- 일부 키워드는 크롤러가 검색 결과를 무검증으로 신뢰해서(사이트 자체 검색이 반환한 글을 그대로 저장), 완전히 무관한 문서가 특정 키워드로 잘못 태깅되는 사례가 있었습니다 (예: "거제야호" 검색에 수년 전 괌 여행기가 딸려온 사례, "야르"처럼 흔한 감탄사가 무관한 갤러리 글에 다수 매칭된 사례). `search_relevant_chunks`에 길이 필터 + title/text 키워드 하드필터를 넣어 이런 오염을 검색 단계에서 방어하고 있지만, 근본 원인은 크롤링 단계에 있습니다 — 자세한 내용은 `docs/crawler-contamination-findings.md` 참고.
- `SOURCES=None`(전체 소스)으로 열면 tavily발 SEO/콘텐츠 팜 블로그 글이 섞여 들어옵니다. 길이/키워드/dense 필터는 다 통과하지만 신뢰도가 검증 안 된 콘텐츠라, `max_per_source`/`MERGED_MAX_PER_SOURCE`로 쏠림만 막았지 근본 해결은 아직입니다. 이 문제가 심하면 `SOURCES`로 tavily를 다시 제외하세요.

## 실행 중 주의사항

이 노트북을 Jupyter/VS Code 등에서 **열어놓은 채로 내가(Claude) 파일을 직접 수정하면, 에디터의 자동저장이 그 열려있던(수정 전) 버전으로 디스크를 덮어써서 수정사항이 사라질 수 있습니다.** 실제로 이 문제로 8~9번 섹션이 한 번 유실된 적이 있습니다. 노트북 관련 요청을 하기 전에는 편집기에서 이 파일을 닫아두거나, 수정 후 반드시 탭을 새로고침/다시 열어서 디스크의 최신 버전을 불러오세요.

## 1. 환경 설정

**이 셀이 하는 일**: OS 환경변수, 인코딩, import를 준비합니다.

**바꿀 것**: `USE_EC2` — EC2 진짜 데이터를 보려면 `True`(기본값), 로컬 docker에 테스트 데이터를 직접 채웠을 때만 `False`.

**주의**:
- `.env`의 `MONGODB_URI`/`QDRANT_HOST`는 `mongo`/`qdrant`라는 docker-compose 내부 네트워크 호스트명을 가리키며 `mimori-flask` 컨테이너 안에서만 resolve됩니다. `.env` 파일 자체는 건드리지 마세요(도커 컨테이너들이 깨집니다) — 이 셀이 로컬 커널일 때 자동으로 `localhost`로 대체합니다.
- `USE_EC2=True`로 쓰려면 아래 SSH 터널이 먼저 떠 있어야 합니다:
  ```
  ssh -i ~/.ssh/id_ed25519 -L 27017:localhost:27017 -L 36333:localhost:16333 ec2-user@100.29.36.216
  ```
  (mongo는 27017 그대로, qdrant는 EC2용으로 로컬 36333을 씀. `6333`은 Windows가 예약한 포트 범위라 로컬에서 못 열고, `16333`은 로컬 docker qdrant가 이미 쓰고 있어서 겹치지 않게 `36333`을 씀. 원격 타겟도 `6333`이 아니라 `16333`인데, EC2의 `mimori-qdrant` 컨테이너 역시 호스트 16333 → 컨테이너 6333으로 매핑돼 있어서 EC2 자신도 6333을 직접 안 듣기 때문입니다. 타겟을 6333으로 잘못 잡으면 SSH 로그인은 되지만 `Connection refused`가 뜹니다.)

In [ ]:
import sys
import os

os.environ.setdefault("KMP_DUPLICATE_LIB_OK", "TRUE")

# 이 커널이 docker 컨테이너 밖(로컬)에서 실행 중이면 .env의 mongo/qdrant 호스트명이
# DNS로 안 풀립니다. 컨테이너 안(env_file로 이미 mongo/qdrant가 주입된 상태)에서 실행
# 중이면 아래 setdefault는 아무 효과가 없고, 로컬 커널일 때만 값이 대체됩니다.
#
# USE_EC2 = True: EC2 서버(진짜 운영 데이터)에 SSH 터널로 붙습니다.
#   ssh -i ~/.ssh/id_ed25519 -L 27017:localhost:27017 -L 36333:localhost:16333 ec2-user@100.29.36.216
#   (mongo는 27017 그대로, qdrant는 EC2용으로 로컬 36333을 씀 — 로컬 docker qdrant가 16333을 이미
#    쓰고 있고, 6333 자체는 Windows가 예약한 포트 범위라 로컬에서 못 엽니다. 원격 타겟도 6333이 아니라
#    16333인데, EC2의 mimori-qdrant 컨테이너도 호스트 16333 -> 컨테이너 6333으로 매핑돼 있어서
#    EC2 자신도 6333을 직접 안 듣기 때문입니다. 타겟을 6333으로 잘못 잡으면 SSH 로그인은 되지만
#    포워딩에서 "Connection refused"가 납니다.)
# USE_EC2 = False: 로컬 docker-compose qdrant(16333)를 봅니다. 로컬에서 직접 크롤링/전처리/
#   임베딩을 돌려서 테스트 데이터를 채웠을 때만 씁니다.
USE_EC2 = True

os.environ.setdefault("MONGODB_URI", "mongodb://localhost:27017")
os.environ.setdefault("QDRANT_HOST", "localhost")
os.environ["QDRANT_PORT"] = "36333" if USE_EC2 else "16333"

# 이 노트북은 analysis/ 안에 있으므로 커널 작업 디렉토리는 analysis/ 입니다.
# 프로젝트 루트(analysis/의 상위 폴더)를 import 경로에 추가합니다.
PROJECT_ROOT = os.path.abspath("..")
if PROJECT_ROOT not in sys.path:
    sys.path.insert(0, PROJECT_ROOT)

if sys.platform == "win32":
    try:
        sys.stdin.reconfigure(encoding="utf-8")
        sys.stdout.reconfigure(encoding="utf-8")
        sys.stderr.reconfigure(encoding="utf-8")
    except AttributeError:
        pass

from analysis.pipeline import list_analyzable_keywords
# facet 검색·병합·프롬프트 조립은 rag_pipeline의 공유 함수를 쓴다 — 4번 셀(단일 키워드)과
# 8번 셀(일괄 평가)에 인라인으로 중복돼 있던 로직을 프로덕션 모듈로 추출한 것.
from analysis.rag_pipeline import (
    build_facet_prompt,
    default_facet_config,
    encode_facets,
    facet_search,
    search_relevant_chunks,
)
from embedding.encoder import encode_batch
from config.config_cilent import NIM_KEY
from langchain_nvidia_ai_endpoints import ChatNVIDIA

print("설정 완료 (EC2 모드)" if USE_EC2 else "설정 완료 (로컬 모드)")

## 2. 키워드 선택

**이 셀이 하는 일**: MongoDB에서 임베딩이 끝난 밈 키워드 목록을 가져와 보여줍니다.

**실험하려면**: 출력된 목록 중 하나를 골라 `KEYWORD` 변수에 문자열로 직접 대입한 뒤 재실행하세요.

In [ ]:
keywords = list_analyzable_keywords()
print(f"분석 가능한 키워드 {len(keywords)}개:")
for kw in keywords:
    print(f"  - {kw}")

KEYWORD = "야르"
print(f"\n현재 선택된 KEYWORD = {KEYWORD!r}")

## 2.5. z-score / 유행 상태 확인

**이 셀이 하는 일**: 선택한 키워드(`KEYWORD`)의 최근 검색량 기반 유행 상태를 네이버 데이터랩에서 조회합니다.

**실험하려면**: 아래 코드 셀의 `INCLUDE_TREND`를 `True`/`False`로 바꾼 뒤, 이 셀을 먼저 재실행하고 "6. 프롬프트 작성" 셀을 실행해야 반영됩니다(`INCLUDE_TREND`는 이 셀에서 정의되므로, 값만 바꾸고 이 셀을 다시 실행하지 않으면 이전 값이 그대로 남아있습니다).

In [ ]:
from trend.trend_service import format_trend_context

trend_info = format_trend_context(KEYWORD)
if trend_info:
    print(trend_info)
else:
    print("z-score/유행 상태를 가져오지 못했습니다 (NAVER API 키 미설정이거나 데이터 없음).")

INCLUDE_TREND = True

## 3. 질문(facet) 설정 + 임베딩

**이 셀이 하는 일**: 밈 하나를 한 각도(예: "왜 유행했나요?")로만 묻지 않고, **의미/유행_이유/사용법/사용자층** 네 각도(facet)로 나눠 각각 질문을 만들고, BGE-M3로 한 번에 배치 임베딩합니다.

**왜 facet을 나누나**: 단일 질문 하나로 검색하면 그 질문 방향에 편중된 청크만 top_k를 채우게 됩니다. 각도를 나눠 각각 검색하면(4번 셀), 한 질문으로는 안 뽑히던 정보(예: 사용법 예시)도 다른 질문에서 뽑힐 수 있습니다.

**FACET_CONFIG의 각 값**:
| 키 | 의미 |
|---|---|
| `question` | 그 facet을 검색할 때 쓰는 질문 텍스트. `f"{KEYWORD}, ..."`처럼 쉼표로 키워드를 붙여, 은/는 조사 활용(받침 유무·특수문자 키워드 문제) 없이 sparse(lexical) 검색이 키워드 토큰을 인식하게 합니다. |
| `top_k` | 이 facet에서 최종적으로 가져올 청크 개수 |
| `min_length` | 이 facet에서 허용할 청크 최소 길이(글자 수). 짧으면 정보가 없다고 보고 버립니다. **"사용법" facet을 한때 10으로 낮춰봤는데, "야르"처럼 흔한 감탄사는 노이즈("야르" 한 줄짜리 게시글)도 똑같이 짧아서 노이즈만 더 들어오는 역효과가 확인돼 30으로 되돌렸습니다.** |
| `over_fetch_factor` | Qdrant에서 실제로는 `top_k * over_fetch_factor`개까지 넉넉히 받아온 뒤, 아래 필터들을 통과한 것만 추려 최종 `top_k`개를 채웁니다. |
| `min_dense_score` | RRF 융합 점수는 순위 기반이라 절대적 관련성 척도가 아니라서, 원본 dense 코사인 유사도에 **별도 하한선**을 둡니다. "야르 + 무관한 댓글이 붙어 길이 필터를 우연히 통과한 청크"처럼 리터럴 매치+노이즈 조합을 걸러내려는 목적. `0`이면 비활성. sparse로만 강하게 매치된 진짜 관련 청크가 이 하한선 때문에 같이 걸러질 수 있으니, 5번 셀 결과를 보며 조심스럽게 조정하세요(리랭커 도입 전까지의 임시방편입니다). |
| `max_per_source` | 한 소스가 top_k를 독점하지 못하게 소스당 최대 개수를 제한합니다. `SOURCES=None`(전체 소스)로 열어보니, tavily 블로그 글이 키워드를 반복적으로 잘 써놔서 필터를 다 통과하며 top_k를 독점하는 현상이 확인돼 추가했습니다. **주의**: 이건 "소스 다양성"만 강제할 뿐 "신뢰도"를 판단하는 게 아닙니다 — 여전히 SEO 콘텐츠 팜 글이 몇 개는 섞여 들어올 수 있고, 이 문제는 아직 별도 해법이 필요합니다(자세한 내용은 이 대화의 tavily 오염 관련 논의 참고). |

**실험하려면**: 이 표의 값들을 자유롭게 바꾸세요 — facet 추가/삭제, 질문 문구 수정, facet별 파라미터 조정 전부 이 셀 하나에서 끝납니다. 바꾼 뒤에는 이 셀부터 다시 실행해야 4번 셀(검색)에 반영됩니다.

In [ ]:
# 기본값은 rag_pipeline.default_facet_config(KEYWORD)와 동일하다. 여기서는 실험을 위해
# 값을 바로 바꿔볼 수 있도록 인라인으로 펼쳐둔 것 — 프로덕션 기본값을 바꾸려면
# default_facet_config 쪽을 수정해야 한다.
FACET_CONFIG = {
    "의미":     {"question": f"{KEYWORD}, 무슨 의미?",           "top_k": 5, "min_length": 30, "over_fetch_factor": 3, "min_dense_score": 0.3, "max_per_source": 2},
    "유행_이유": {"question": f"{KEYWORD}, 왜 유행했나요?",       "top_k": 5, "min_length": 30, "over_fetch_factor": 3, "min_dense_score": 0.3, "max_per_source": 2},
    "사용법":   {"question": f"{KEYWORD}, 어떻게 사용하나요?",    "top_k": 5, "min_length": 30, "over_fetch_factor": 3, "min_dense_score": 0.3, "max_per_source": 2},
    "사용자층": {"question": f"{KEYWORD}, 주로 누가 사용하나요?", "top_k": 5, "min_length": 30, "over_fetch_factor": 3, "min_dense_score": 0.3, "max_per_source": 2},
}

_facet_names = list(FACET_CONFIG.keys())

# 4개 질문을 한 번에 배치 임베딩 (모델을 4번 따로 부르지 않음) — rag_pipeline.encode_facets 공유
FACET_VECTORS = encode_facets(FACET_CONFIG)

for name in _facet_names:
    cfg = FACET_CONFIG[name]
    print(f"[{name}] {cfg['question']!r}  (top_k={cfg['top_k']}, min_length={cfg['min_length']}, over_fetch_factor={cfg['over_fetch_factor']}, min_dense_score={cfg['min_dense_score']}, max_per_source={cfg['max_per_source']})")
print(f"dense_vec 길이 = {len(FACET_VECTORS[_facet_names[0]]['dense'])}")

## 4. 검색 파라미터 + 실행

**이 셀이 하는 일**: 3번 셀에서 만든 facet(의미/유행_이유/사용법/사용자층)마다 검색하고 하나의 컨텍스트로 병합하는 전 과정을 `rag_pipeline.facet_search()` 호출 한 번으로 실행합니다. **예전에 이 셀에 인라인으로 있던 병합/근접중복/소스캡 로직은 8번 셀(일괄 평가)과 글자 그대로 중복돼 있어서 `analysis/rag_pipeline.py`로 추출했습니다** — 프로덕션(`rag_main.py`의 facet 모드)과 8번 셀 모두 같은 함수를 씁니다. 내부 동작:

1. facet마다 `search_relevant_chunks` 호출: `top_k * over_fetch_factor`개까지 넉넉히 후보를 받아온 뒤(RRF 융합), **길이가 `min_length` 미만이면 제외**, **title/text 어디에도 (공백·물결표 무시) 키워드가 없으면 제외**, **`min_dense_score` 설정 시 원본 dense 코사인 유사도가 그 값 미만이어도 제외**. 검증 통과작 중 소스당 `max_per_source`개까지만 우선 채우고, 부족하면 한도를 풀어서 — 그래도 부족하면 검증 실패작으로 백필.
2. 4개 facet 결과를 **point.id 기준으로 합치고 중복 제거**, 추가로 **`difflib`로 근접 중복**(다른 URL이지만 내용이 사실상 같은 청크, 예: 네이버블로그 `in.naver.com`/`blog.naver.com` 미러링)도 제외.
3. **`MERGED_MAX_PER_SOURCE`로 facet 전체 합산 기준 소스 상한을 한 번 더 적용** — `max_per_source`는 facet 하나 안에서만 걸리기 때문에, facet 4개가 전부 같은 소스에서 2개씩 뽑으면 합계가 8개까지 쏠릴 수 있다는 게 실제로 확인돼서("킹받네" 키워드에서 8개 전부 tavily) 추가했습니다. 상한 때문에 컨텍스트가 `MIN_MERGED_TOTAL`보다 적어지면, 상한을 넘겨서라도 그만큼은 채웁니다.

반환된 `merged_points`가 6번 셀(프롬프트)에 그대로 들어가고, `facet_points`/`point_facets`는 5번 셀(진단 출력)이 씁니다.

**실험하려면**:
- `SOURCES`로 검색 대상 소스 자체를 제한할 수도 있습니다(예: `["dcinside", "natepann", "namuwiki"]`로 tavily 제외). 지금은 `None`(전체 소스)으로 열어두고, 소스 쏠림은 `max_per_source`/`MERGED_MAX_PER_SOURCE`로, 콘텐츠 중복은 근접 중복 탐지로 다루는 실험 중입니다. **단, 이 필터들은 "다양성"만 강제할 뿐 tavily 블로그 콘텐츠 자체의 신뢰도 문제는 해결하지 못합니다** — 그 문제가 심하면 `SOURCES`로 아예 제한하는 게 더 확실합니다.
- `_NEAR_DUP_THRESHOLD`(기본 0.8)를 낮추면 더 느슨하게(비슷하기만 해도) 중복 처리, 높이면 더 엄격하게(거의 똑같아야) 중복 처리합니다.
- `MERGED_MAX_PER_SOURCE`(기본 6)/`MIN_MERGED_TOTAL`(기본 8)도 이 셀에서 바로 조정합니다. (프로덕션 기본값은 `config/config_cilent.py`의 `RAG_FACET_*` 상수.)
- facet별 `top_k`/`min_length`/`over_fetch_factor`/`min_dense_score`/`max_per_source`는 여기가 아니라 **3번 셀의 `FACET_CONFIG`**에서 바꿉니다.

In [ ]:
SOURCES = None  # 전체 소스. 소스 쏠림은 SOURCES 자체를 제한하는 대신 아래 max_per_source/MERGED_MAX_PER_SOURCE로 방지.
_NEAR_DUP_THRESHOLD = 0.8  # 이 이상 유사하면 같은 내용 재게시로 보고 제외 (예: 네이버블로그 in.naver.com / blog.naver.com 미러링)
MERGED_MAX_PER_SOURCE = 6  # 병합(facet 전체 합산) 기준 소스당 상한 — FACET_CONFIG의 max_per_source는 facet마다 따로 걸려서, facet 4개가 전부 같은 소스를 2개씩 뽑으면 합계가 8개까지 쏠릴 수 있음. 최종 병합 단계에서 한 번 더 제한.
MIN_MERGED_TOTAL = 8  # 소스 상한 때문에 컨텍스트가 너무 비지 않도록 최소 이만큼은 채움(부족하면 상한 넘겨서라도 보충)

# 검색 + 병합/근접중복/병합소스캡 전 과정은 rag_pipeline.facet_search()가 수행한다
# (8번 셀의 일괄 평가와 완전히 같은 코드 경로 — 예전에 두 셀에 중복돼 있던 로직을 추출).
# FACET_VECTORS를 넘겨 3번 셀에서 이미 만든 임베딩을 재사용한다(재임베딩 방지).
merged_points, _diag = facet_search(
    KEYWORD, FACET_CONFIG, facet_vectors=FACET_VECTORS, sources=SOURCES,
    merged_max_per_source=MERGED_MAX_PER_SOURCE,
    min_merged_total=MIN_MERGED_TOTAL,
    near_dup_threshold=_NEAR_DUP_THRESHOLD,
)
facet_points = _diag["facet_points"]  # facet_name -> 필터링 적용된 최종 top_k 청크 리스트 (5번 셀용)
point_facets = _diag["point_facets"]  # point.id -> 그 청크가 뽑힌 facet 이름들 (5번 셀 진단용)

print(f"SOURCES = {SOURCES!r}")
for name in _facet_names:
    src_counts = {}
    for p in facet_points[name]:
        src = p.payload.get("source")
        src_counts[src] = src_counts.get(src, 0) + 1
    print(f"  [{name}] {len(facet_points[name])}개  소스분포={src_counts}")
print(f"근접 중복으로 제외: {_diag['near_dup_skipped']}개")
print(f"병합 소스 상한으로 보류: {_diag['deferred']}개")
print(f"최종 컨텍스트: {len(merged_points)}개  소스분포={_diag['source_counts']}")

## 5. 검색 결과 확인 (facet별 + 최종 병합 결과 + 노이즈 비율 요약)

**이 셀이 하는 일**: facet(의미/유행_이유/사용법/사용자층)마다 검색된 청크를 따로 보여준 뒤, 마지막에 중복 제거된 최종 컨텍스트(`merged_points`)를 보여주고, **그중 몇 %가 검증(길이/키워드)에 실패해 백필된 것으로 의심되는지 숫자로 요약**합니다. 각 줄 앞:
- `[facet1,facet2]`: 이 청크가 어느 facet(들)에서 뽑혔는지 (여러 facet에 걸치면 콤마로 나열)
- `⚠️길이/키워드 미달(...)`: `search_relevant_chunks`의 필터를 통과 못 했을 가능성이 높은 청크(백필로 채워졌을 가능성). **이게 자주 뜨면 그 facet의 실제 문서 풀이 얕다는 뜻**이니, 3번 셀에서 `over_fetch_factor`를 높이거나 `min_length`/`min_dense_score`를 낮춰보세요.

**맨 아래 `[요약]` 줄**: "최종 컨텍스트 N개 중 ⚠️ M개 (X%)" — 이 숫자를 실험할 때마다 기록해두면, 설정을 바꾼 게 정말 노이즈를 줄였는지 눈대중이 아니라 수치로 비교할 수 있습니다.

**실험하려면**: 이 셀 자체는 결과를 출력만 하므로 수정할 것이 없습니다 — 3번 셀(FACET_CONFIG)과 4번 셀(SOURCES)을 바꾸고 재실행해서 이 요약 숫자가 어떻게 변하는지 비교하세요.

In [ ]:
def _norm_kw(s):
    return s.replace(" ", "").replace("~", "")


def _is_flagged(p, facets):
    """search_relevant_chunks의 검증(길이/키워드)을 청크 하나에 대해 재현.
    facets가 여러 개면 그중 가장 관대한(작은) min_length를 기준으로 판단 —
    "이 청크가 통과할 수 있었던 가장 쉬운 조건에서도 실패했는지"를 본다."""
    text = p.payload.get("text", "")
    title = p.payload.get("title", "")
    if _norm_kw(KEYWORD) not in _norm_kw(text + title):
        return True
    min_len = min((FACET_CONFIG[f]["min_length"] for f in facets), default=30)
    return len(text.strip()) < min_len


def _print_points(label, points, point_facets=None):
    print(f"--- {label} ({len(points)}개) ---")
    warned = 0
    for p in points:
        title = p.payload.get("title") or "제목 없음"
        url = p.payload.get("url") or "출처 없음"
        text = p.payload.get("text", "")

        facets = point_facets.get(p.id, []) if point_facets else []
        facet_tag = "[" + ",".join(facets) + "] " if facets else ""

        flagged = _is_flagged(p, facets)
        warn = "  ⚠️길이/키워드 미달(백필된 청크 의심)" if flagged else ""
        if flagged:
            warned += 1

        print(f"  {facet_tag}score={p.score:.4f} | {title} ({url}){warn}")
        print(f"  {text[:150]}")
    print()
    return warned


for name in _facet_names:
    _print_points(f"FACET: {name}", facet_points[name], point_facets={p.id: [name] for p in facet_points[name]})

warned_count = _print_points("최종 컨텍스트 (중복 제거됨, 6번 셀에 그대로 전달)", merged_points, point_facets)
if merged_points:
    print(f"[요약] 최종 컨텍스트 {len(merged_points)}개 중 ⚠️(백필 의심) {warned_count}개 ({warned_count / len(merged_points) * 100:.0f}%)")
else:
    print("[요약] 최종 컨텍스트 없음")

## 6. 프롬프트 작성

**이 셀이 하는 일**: 병합된 청크(`merged_points`)와 유행 상태 정보(`trend_info`, `INCLUDE_TREND`가 `True`일 때만)를 근거 자료로 넣어 LLM에게 실제로 전달할 프롬프트를 완성합니다. 자유 질문 하나가 아니라, **의미/유행 이유/사용법/사용자층 네 항목을 각각 답하도록 고정 지시**하는 프롬프트입니다(3번 셀의 facet 구성과 대응).

**템플릿은 `analysis/rag_facet_prompt_template.md`로 승격됐습니다** — 이 노트북에 인라인으로 있던 `PROMPT_TEMPLATE`을 정식 파일로 옮긴 것으로, 프로덕션(`rag_main.py`의 facet 모드)과 같은 파일을 씁니다. 조립은 `rag_pipeline.build_facet_prompt()`가 담당하며, 각 자료 앞에 `[출처: 제목 / URL / 소스유형: dcinside|natepann|namuwiki|tavily|youtube]`를 붙여 LLM이 **1차 자료(커뮤니티 반응)와 2차 자료(tavily/namuwiki 같은 가공된 설명형 콘텐츠)를 구분**하게 합니다 — tavily 블로그 글이 서로 다른 기원설을 지어내는 사례가 확인돼서, 2차 자료에만 나오는 구체적 주장은 약하게 표현하도록 지시합니다. "숫자·고유명사 등은 원문에 실제로 있을 때만 쓰라"는 지시도 포함돼 있습니다.

**실험하려면**: `analysis/rag_facet_prompt_template.md` 파일을 직접 수정한 뒤 이 셀을 재실행하세요 — `build_facet_prompt`가 매번 파일을 다시 읽으므로 커널 재시작이 필요 없습니다. `{keyword}` / `{trend_info}` / `{context}` 세 자리표시자는 반드시 그대로 남겨둬야 합니다. `.format()`을 쓰므로, 프롬프트에 `{`/`}` 자체를 문자로 넣고 싶으면(예: JSON 출력 형식을 지시하는 경우) `{{`/`}}`로 이스케이프해야 합니다. **주의**: 이제 이 파일은 프로덕션도 같이 쓰는 정식 템플릿이므로, 실험으로 바꾼 내용을 커밋하기 전에 의도한 변경인지 확인하세요(실험만 하고 되돌릴 거면 git으로 복원).

In [ ]:
# 프롬프트 템플릿은 analysis/rag_facet_prompt_template.md로 승격됨 (프로덕션 rag_main.py의
# facet 모드와 같은 파일 사용). 문구를 실험하려면 그 파일을 수정하고 이 셀을 재실행하세요 —
# build_facet_prompt가 매번 파일을 다시 읽습니다.
#
# trend_info: 2.5번 셀에서 만든 값을 그대로 쓴다. INCLUDE_TREND=False면 빈 문자열을
# 명시적으로 넘겨 build_facet_prompt가 내부에서 다시 조회(네트워크 호출)하지 않게 한다.
prompt = build_facet_prompt(
    KEYWORD,
    merged_points,
    trend_info=((trend_info or "") if INCLUDE_TREND else ""),
)
print(prompt)

## 7. LLM 호출

**이 셀이 하는 일**: 완성된 프롬프트를 NVIDIA API(ChatNVIDIA)로 보내고 답변을 받아 출력합니다.

**실험하려면**: `MODEL` / `TEMPERATURE` / `TOP_P` 값을 바꿔서, 같은 프롬프트에도 답변이 어떻게 달라지는지 비교해보세요.

In [ ]:
assert NIM_KEY, "NIM_KEY가 .env에 설정되어 있지 않습니다"

from analysis.pipeline import invoke_with_retry

MODEL = "deepseek-ai/deepseek-v4-flash"
TEMPERATURE = 0.3  # 1 -> 0.3: 근거자료 기반 요약/분석 태스크라 확신도 높은 답 위주로(할루시네이션 억제, 실험 재현성 확보)
TOP_P = 0.95

llm_client = ChatNVIDIA(
    model=MODEL,
    api_key=NIM_KEY,
    temperature=TEMPERATURE,
    top_p=TOP_P,
    max_completion_tokens=16384,
    timeout=6000,
)

# 503(서버 혼잡) 등 일시적 오류는 지수 백오프로 자동 재시도하고 진행 상황을 출력한다.
response = invoke_with_retry(llm_client, [{"role": "user", "content": prompt}])
print(response.content)

## 8. 여러 키워드 일괄 검증 (평가 하네스)

**이 셀이 하는 일**: 지금까지 넣은 설정(facet 구성, 길이/키워드/dense/소스캡/근접중복 필터)이 "야르" 하나가 아니라 **여러 키워드에서도 일관되게 노이즈를 줄이는지** 확인합니다. 2~7번 셀의 대화형 실험(`KEYWORD`, `FACET_CONFIG`, `merged_points` 등)과는 완전히 독립적으로 동작해서, 이 셀을 실행해도 위쪽 셀 변수들은 안 건드립니다.

**읽는 법**: 키워드별로 최종 컨텍스트 개수, ⚠️(백필 의심) 개수·비율, 근접 중복 제외 개수, 소스 분포가 표로 나오고, 맨 아래 전체 평균 ⚠️ 비율이 나옵니다. 지금까지는 "야르" 하나로만 46% → (필터 적용 후) 확인했는데, 이 표로 다른 키워드(복잡한 밈 "거제야호", 특수문자 포함 "야호~"/"좋~다~", 사람 이름 "홍명보", 짧은 감탄사 "킹받네")에서도 비슷하게 낮은지 볼 수 있습니다.

**실험하려면**: `TEST_KEYWORDS`(테스트할 키워드 목록)와 `EVAL_SOURCES`(소스 제한)를 바꾸세요. 설정(`min_length`/`min_dense_score`/`max_per_source` 등)을 바꿔가며 이 셀을 재실행하면, "이 변경이 여러 키워드에 걸쳐 실제로 개선인지" 숫자로 비교할 수 있습니다. **주의**: 키워드마다 facet 4개 × Qdrant 쿼리 여러 번을 돌리므로, 키워드 수를 늘리면 그만큼 오래 걸립니다.

In [ ]:
import json
import os
import datetime

TEST_KEYWORDS = ["야르", "거제야호", "야호~", "좋~다~", "홍명보", "킹받네"]
EVAL_SOURCES = None  # 4번 셀과 동일하게 전체 소스. 필요하면 ["dcinside","natepann","namuwiki"]처럼 제한.
EVAL_MERGED_MAX_PER_SOURCE = 6  # 4번 셀의 MERGED_MAX_PER_SOURCE와 동일한 개념
EVAL_MIN_MERGED_TOTAL = 8  # 4번 셀의 MIN_MERGED_TOTAL과 동일한 개념
EVAL_LOG_PATH = "eval_runs.jsonl"  # 실행 기록 누적 파일 (analysis/ 안에 쌓임, 노트북 cwd 기준)


def _build_facet_config_for(keyword: str) -> dict:
    """기본은 프로덕션과 동일한 rag_pipeline.default_facet_config. 평가에서 파라미터를
    바꿔 실험하려면 반환 전에 값을 수정하세요 (예: cfg["의미"]["min_length"] = 20)."""
    return default_facet_config(keyword)


def _norm_kw_eval(s):
    return s.replace(" ", "").replace("~", "")


def _is_flagged_eval(p, facets, facet_config, keyword):
    text = p.payload.get("text", "")
    title = p.payload.get("title", "")
    if _norm_kw_eval(keyword) not in _norm_kw_eval(text + title):
        return True
    min_len = min((facet_config[f]["min_length"] for f in facets), default=30)
    return len(text.strip()) < min_len


def evaluate_keyword(keyword: str, sources=None) -> dict:
    """FACET_CONFIG/merged_points 등 대화형 전역변수를 전혀 건드리지 않고,
    한 키워드에 대해 facet 검색 -> 병합/중복제거/병합소스캡 -> 노이즈 비율까지
    독립적으로 계산. 검색+병합은 4번 셀과 완전히 같은 rag_pipeline.facet_search()를
    호출한다 — 예전엔 이 함수 안에 4번 셀과 같은 로직이 인라인으로 중복돼 있었다."""
    facet_config = _build_facet_config_for(keyword)

    # facet_vectors를 안 넘기면 facet_search가 내부에서 encode_facets로 임베딩한다
    # (예전의 키워드별 encode_batch 호출과 동일한 동작).
    merged, diag = facet_search(
        keyword, facet_config, sources=sources,
        merged_max_per_source=EVAL_MERGED_MAX_PER_SOURCE,
        min_merged_total=EVAL_MIN_MERGED_TOTAL,
    )

    warned = sum(
        1 for p in merged
        if _is_flagged_eval(p, diag["point_facets"].get(p.id, []), facet_config, keyword)
    )

    return {
        "keyword": keyword,
        "total": len(merged),
        "warned": warned,
        "warned_pct": round(warned / len(merged) * 100, 1) if merged else 0.0,
        "near_dup_skipped": diag["near_dup_skipped"],
        "source_counts": diag["source_counts"],
    }


def save_eval_run(results: list[dict], path: str = EVAL_LOG_PATH) -> None:
    """이번 실행(설정 스냅샷 + 키워드별 결과)을 한 줄(JSON)로 파일에 이어붙인다.
    검색 단계는 결정적이라(같은 Qdrant 데이터 + 같은 코드면 결과가 항상 같음),
    이 로그를 시간순으로 쌓아두면 "그 사이 설정을 바꿨는지/크롤러가 다시 돌았는지"를
    타임스탬프로 짚어가며 실행 간 비교를 할 수 있다."""
    config_snapshot = {
        "eval_sources": EVAL_SOURCES,
        "merged_max_per_source": EVAL_MERGED_MAX_PER_SOURCE,
        "min_merged_total": EVAL_MIN_MERGED_TOTAL,
        "facet_params": {
            name: {k: v for k, v in cfg.items() if k != "question"}
            for name, cfg in _build_facet_config_for("__sample__").items()
        },
    }
    record = {
        "timestamp": datetime.datetime.now().isoformat(timespec="seconds"),
        "config": config_snapshot,
        "results": results,
    }
    with open(path, "a", encoding="utf-8") as f:
        f.write(json.dumps(record, ensure_ascii=False) + "\n")


results = [evaluate_keyword(kw, sources=EVAL_SOURCES) for kw in TEST_KEYWORDS]

print(f"{'키워드':<10} {'총':>4} {'⚠️':>4} {'⚠️%':>7} {'근접중복':>8}  소스분포")
print("-" * 70)
for r in results:
    print(f"{r['keyword']:<10} {r['total']:>4} {r['warned']:>4} {r['warned_pct']:>6.1f}% {r['near_dup_skipped']:>8}  {r['source_counts']}")

avg_pct = sum(r["warned_pct"] for r in results) / len(results) if results else 0
print("-" * 70)
print(f"[전체 평균] ⚠️ 비율 {avg_pct:.1f}% (키워드 {len(results)}개 기준)")

save_eval_run(results)
print(f"\n[저장됨] {os.path.abspath(EVAL_LOG_PATH)}")

## 9. 저장된 실행 기록 비교

**이 셀이 하는 일**: 8번 셀이 실행마다 `eval_runs.jsonl`에 이어붙인 기록을 불러와, 키워드 × 실행시각 표로 보여줍니다. 검색 단계는 결정적이라(같은 Qdrant 데이터 + 같은 코드면 결과가 항상 동일), 이 표에서 어떤 키워드의 ⚠️%가 실행 시점마다 달라졌다면 **그 사이 설정(3~4번 셀 또는 8번 셀 상단)을 바꿨거나, 크롤러가 다시 돌아서 데이터가 늘었다는 뜻**입니다.

**실험하려면**: 8번 셀에서 설정을 바꾸고 재실행할 때마다 기록이 한 줄씩 쌓입니다. 이 셀을 다시 실행하면 최신 기록까지 포함해서 표를 다시 그립니다. `print_eval_history(last_n=N)`으로 최근 N개 실행만 볼 수 있습니다.

In [ ]:
def load_eval_runs(path: str = EVAL_LOG_PATH) -> list[dict]:
    if not os.path.exists(path):
        return []
    runs = []
    with open(path, encoding="utf-8") as f:
        for line in f:
            line = line.strip()
            if line:
                runs.append(json.loads(line))
    return runs


def print_eval_history(last_n: int = 5, path: str = EVAL_LOG_PATH) -> None:
    runs = load_eval_runs(path)[-last_n:]
    if not runs:
        print(f"저장된 실행 기록이 없습니다 ({path}). 먼저 8번 셀을 실행하세요.")
        return

    all_keywords = []
    for run in runs:
        for r in run["results"]:
            if r["keyword"] not in all_keywords:
                all_keywords.append(r["keyword"])

    col_width = 10
    header = "실행 시각".ljust(20) + "".join(kw.ljust(col_width) for kw in all_keywords)
    print(header)
    print("-" * len(header))
    for run in runs:
        pct_by_kw = {r["keyword"]: r["warned_pct"] for r in run["results"]}
        row = run["timestamp"].ljust(20)
        for kw in all_keywords:
            val = pct_by_kw.get(kw)
            row += (f"{val:.0f}%".ljust(col_width) if val is not None else "-".ljust(col_width))
        print(row)

    print(f"\n(총 {len(load_eval_runs(path))}개 실행 기록 중 최근 {len(runs)}개 표시, 파일: {os.path.abspath(path)})")


print_eval_history()